In [ ]:
import os
import numpy as np
import csv

# 1. 데이터 로드 (npz 파일에 X, y가 저장되어 있다고 가정)
npz_path = r"C:\Users\daeun\Desktop\학부연구생 과제\20250204\QG_jets.npz"
data = np.load(npz_path)
X = data['X']   # shape: (num_total_jets, M, 4)

# 2. 전처리할 제트 수와 particle_counts 배열 생성
num_images = 20000  # 전처리할 제트 수
particle_counts = np.zeros((num_images, 1), dtype=np.float32)

for i in range(num_images):
    jet = X[i]          # shape: (M, 4)
    pts = jet[:, 0]     # pT 값들
    valid = pts > 0     # 유효 입자 판별 (pT > 0 인 입자)
    particle_counts[i, 0] = np.sum(valid)  # 유효 입자 개수 저장

# 3. 저장 경로 설정
dataset_dir = r"C:\Users\daeun\Desktop\학부연구생 과제\20250204\dataset"
if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)

# 4. 저장 방식 : CSV 파일로 저장
csv_save_path = os.path.join(dataset_dir, "particle_counts.csv")
with open(csv_save_path, mode='w', newline='') as csv_file:
    writer = csv.writer(csv_file)
    # 헤더 작성
    writer.writerow(["jet_index", "particle_count"])
    # 각 제트의 인덱스와 유효 입자 개수 기록
    for i, count in enumerate(particle_counts):
        writer.writerow([i, int(count[0])])
print(f"CSV 파일로 저장 완료: {csv_save_path}")


CSV 파일로 저장 완료: C:\Users\daeun\Desktop\학부연구생 과제\20250204\dataset\particle_counts.csv


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm  # pip install timm

# -----------------------------
# 1. 사용자 정의 멀티모달 Dataset 클래스 (이미지, 입자 개수, 레이블)
# -----------------------------
class JetHeatmapMultiModalDataset(Dataset):
    def __init__(self, labels_csv, particle_counts_csv, root_dir, transform=None):
        """
        Args:
            labels_csv (str): 이미지 파일명과 레이블이 기록된 CSV 파일 경로. (컬럼: "filename", "label")
            particle_counts_csv (str): 입자 개수가 기록된 CSV 파일 경로. (컬럼: "jet_index", "particle_count")
            root_dir (str): 이미지 파일들이 저장된 디렉터리 경로.
            transform (callable, optional): 이미지에 적용할 변환.

        **주의:** 두 CSV 파일의 행 순서가 동일하다고 가정합니다.
        """
        # labels.csv 로드
        self.labels_df = pd.read_csv(labels_csv)
        # particle_counts.csv 로드
        particle_df = pd.read_csv(particle_counts_csv)

        # 두 데이터프레임의 행 순서가 동일하다고 가정하여, particle_count 컬럼을 추가합니다.
        # (만약 공통 key가 있다면 merge할 수 있습니다.)
        self.labels_df['particle_count'] = particle_df['particle_count']

        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        # labels_df의 컬럼: "filename", "label", "particle_count"
        filename = self.labels_df.iloc[idx]['filename']
        img_path = os.path.join(self.root_dir, filename)
        image = Image.open(img_path).convert('RGB')
        label = int(self.labels_df.iloc[idx]['label'])
        # 입자 개수는 float 값으로 변환, 텐서 shape: (1,)
        particle_count = float(self.labels_df.iloc[idx]['particle_count'])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor([particle_count], dtype=torch.float32), label

# -----------------------------
# 2. 멀티모달 모델 정의 (ViT 백본 + 입자 개수 브랜치)
# -----------------------------
class ParticleBranch(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=16, output_dim=8):
        super(ParticleBranch, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class MultiModalViT(nn.Module):
    def __init__(self, num_classes=2, num_trainable_blocks=4):
        super(MultiModalViT, self).__init__()
        # 1. ViT 백본: timm의 사전 학습된 vit_tiny_patch16_224 사용 (분류 헤드는 제거)
        self.vit = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=0)
        # vit_tiny_patch16_224의 embed_dim (출력 차원)은 timm 모델의 속성으로 확인 가능 (일반적으로 192)
        self.vit_feature_dim = self.vit.embed_dim

        # 2. 입자 개수 브랜치 (스칼라 입력을 받아 8차원 특징 벡터 출력)
        self.particle_branch = ParticleBranch(input_dim=1, hidden_dim=16, output_dim=8)

        # 3. ViT 백본의 후반 num_trainable_blocks 블록만 파인튜닝하도록 설정 (나머지는 동결)
        total_blocks = len(self.vit.blocks)
        for i, block in enumerate(self.vit.blocks):
            if i < total_blocks - num_trainable_blocks:
                for param in block.parameters():
                    param.requires_grad = False

        # 4. 최종 분류 헤드: 이미지 특징과 입자 개수 특징을 결합
        combined_dim = self.vit_feature_dim + 8
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, image, particle_count):
        # image: [batch_size, 3, 224, 224]
        # particle_count: [batch_size, 1]
        # ViT 백본을 통해 이미지 특징 추출
        image_features = self.vit(image)  # shape: [batch_size, vit_feature_dim]
        # 입자 개수 브랜치를 통해 특징 추출
        particle_features = self.particle_branch(particle_count)  # shape: [batch_size, 8]
        # 특징 벡터 결합
        combined = torch.cat((image_features, particle_features), dim=1)
        # 최종 분류
        output = self.classifier(combined)
        return output

# -----------------------------
# 3. 메인 코드: 데이터 로드, DataLoader 구성, 학습 루프 등
# -----------------------------
if __name__ == '__main__':
    # GPU 사용 여부 확인
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # 데이터셋 디렉터리 및 CSV 파일 경로 설정
    dataset_dir = r"C:\Users\daeun\Desktop\학부연구생 과제\20250204\dataset"
    labels_csv_path = os.path.join(dataset_dir, "labels.csv")
    particle_counts_csv_path = os.path.join(dataset_dir, "particle_counts.csv")

    # labels.csv와 particle_counts.csv를 결합하여 train/validation 분할 (클래스 비율 유지)
    labels_df = pd.read_csv(labels_csv_path)
    # particle_counts.csv를 로드하여 labels_df에 particle_count 컬럼 추가 (순서가 동일하다고 가정)
    particle_df = pd.read_csv(particle_counts_csv_path)
    labels_df['particle_count'] = particle_df['particle_count']

    print("CSV preview:")
    print(labels_df.head())

    train_df, val_df = train_test_split(labels_df, test_size=0.2, random_state=42, stratify=labels_df['label'])

    # 분할된 데이터를 임시 CSV 파일로 저장 (Dataset 클래스에서 사용)
    train_csv = os.path.join(dataset_dir, "train_labels.csv")
    val_csv = os.path.join(dataset_dir, "val_labels.csv")
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv, index=False)

    # -----------------------------
    # 4. 데이터 변환 및 DataLoader 구성
    # -----------------------------
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        # transforms.Normalize(mean, std) 필요 시 추가
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    # Dataset 객체 생성 (멀티모달: 이미지, 입자 개수, 레이블)
    train_dataset = JetHeatmapMultiModalDataset(labels_csv=train_csv, particle_counts_csv=particle_counts_csv_path,
                                                 root_dir=dataset_dir, transform=train_transform)
    val_dataset = JetHeatmapMultiModalDataset(labels_csv=val_csv, particle_counts_csv=particle_counts_csv_path,
                                               root_dir=dataset_dir, transform=val_transform)

    batch_size = 128
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    # -----------------------------
    # 5. 모델 구성: ViT 백본 + 입자 개수 브랜치, 후반 4개 블록만 파인튜닝
    # -----------------------------
    model = MultiModalViT(num_classes=2, num_trainable_blocks=4)
    model = model.to(device)
    print(model)

    # -----------------------------
    # 6. 손실 함수 및 옵티마이저 설정
    # -----------------------------
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    # -----------------------------
    # 7. 학습 루프
    # -----------------------------
    num_epochs = 30
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total_train = 0

        for images, particle_counts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
            images = images.to(device)                     # [batch_size, 3, 224, 224]
            particle_counts = particle_counts.to(device)   # [batch_size, 1]
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images, particle_counts)  # 출력: (batch_size, 2)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels)
            total_train += labels.size(0)

        train_loss = running_loss / total_train
        train_acc = running_corrects.double() / total_train

        # 검증 단계
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        total_val = 0

        with torch.no_grad():
            for images, particle_counts, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):
                images = images.to(device)
                particle_counts = particle_counts.to(device)
                labels = labels.to(device)
                outputs = model(images, particle_counts)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels)
                total_val += labels.size(0)

        epoch_val_loss = val_loss / total_val
        epoch_val_acc = val_corrects.double() / total_val

        print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

    # -----------------------------
    # 8. 최종 모델 저장
    # -----------------------------
    model_save_path = os.path.join(dataset_dir, "timm_vit_multimodal_finetuned.pth")
    torch.save(model.state_dict(), model_save_path)
    print("Training complete. Final model saved at:", model_save_path)


Using device: cuda
CSV preview:
         filename  label  particle_count
0  jet_000000.png      1              18
1  jet_000001.png      1              17
2  jet_000002.png      1              57
3  jet_000003.png      1              37
4  jet_000004.png      1              35
MultiModalViT(
  (vit): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=Fal

Epoch 1/30 - Validation: 100%|█████████████████████████████████████████████████████████| 32/32 [00:10<00:00,  3.07it/s]


Epoch 1/30: Train Loss: 0.6043, Train Acc: 0.6948 | Val Loss: 0.5344, Val Acc: 0.7283


Epoch 2/30 - Validation: 100%|█████████████████████████████████████████████████████████| 32/32 [00:11<00:00,  2.90it/s]


Epoch 2/30: Train Loss: 0.5188, Train Acc: 0.7487 | Val Loss: 0.5297, Val Acc: 0.7432


...

Epoch 28/30: Train Loss: 0.4452, Train Acc: 0.7974 | Val Loss: 0.4926, Val Acc: 0.7588


Epoch 29/30 - Validation: 100%|████████████████████████████████████████████████████████| 32/32 [00:11<00:00,  2.87it/s]


Epoch 29/30: Train Loss: 0.4418, Train Acc: 0.7976 | Val Loss: 0.4786, Val Acc: 0.7715


Epoch 30/30 - Validation: 100%|████████████████████████████████████████████████████████| 32/32 [00:11<00:00,  2.89it/s]

Epoch 30/30: Train Loss: 0.4434, Train Acc: 0.7991 | Val Loss: 0.4783, Val Acc: 0.7682
Training complete. Final model saved at: C:\Users\daeun\Desktop\학부연구생 과제\20250204\dataset\timm_vit_multimodal_finetuned.pth
